# 11.4 - Memory Systems for Agents

**Module 11 - AI Agents** | Netsetos GenAI Engineering

An LLM is stateless: it only knows what is in its context window right now, so "real" memory is something you engineer around it. This notebook builds all four kinds of agent memory from scratch - a working-memory buffer, summarization/compaction, semantic vector recall, and an episodic experience log - then closes with the context-engineering discipline of deciding what to actually load into a finite, lossy window. Every mechanism runs with no API key.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - install dependencies

The whole lesson runs with no API key, but a couple of demos and the illustrative store snippet reference these packages. Uncomment the line on Colab or a fresh environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install numpy langgraph python-dotenv -q  # noqa


**Explanation**

A one-line dependency install, commented out so a local run with these already installed does nothing.

**How the code works - step by step**
- **`numpy`** - the vector math for the toy embedder in the semantic-memory cell.
- **`langgraph`** - referenced by the illustrative cross-thread Store snippet (not executed).
- **`python-dotenv`** - lets you load API keys from a `.env` file for the optional provider demos.

**In one line:** environment prep, uncomment on a fresh machine.

**What you'll see:** (no output - environment prep)

## Setup - load API keys

None of the runnable memory mechanisms below need a key - they are pure Python. This cell just wires up the environment so the optional multi-provider demos can find a key if you have one, without ever hardcoding it.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Environment plumbing, not a model call. It seeds three provider-key variables to empty strings so later lookups do not crash, and reminds you to supply real keys via the environment or a `.env` file rather than in code.

**How the code works - step by step**
- **`import os`** - access to the process environment.
- **`os.environ.setdefault(...)`** x3 - sets `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and `GOOGLE_API_KEY` to `""` only if they are not already set, so a real key already in your environment is left untouched.
- Any one provider key is enough to start; the multi-provider demos want two or more.

**In one line:** safe key wiring, no key required for this lesson.

**What you'll see:** (no output - environment prep)

## 1 - Classify the four kinds of memory

Before building anything, get the map. Agent memory splits by **duration** (short-term in-context vs long-term in a store) and by **kind** (working / semantic / episodic / procedural). This cell is a tiny classifier that drops each piece of agent state into the right bucket - and every later step builds the mechanism for one of these kinds.

In [ ]:
# THE FOUR KINDS OF AGENT MEMORY - classify what each piece of state IS.
def classify(item):
    kind, span = item["kind"], item["span"]
    if span == "in_context":   return "working",    "the running turn, live in the context window"
    if kind == "fact":         return "semantic",   "a durable fact or preference (long-term store)"
    if kind == "experience":   return "episodic",   "a past event/outcome you can recall (long-term)"
    if kind == "rule":         return "procedural", "a learned instruction the agent follows"
    return "unknown", "?"

items = [
    {"what": "the last 6 chat messages",        "kind": "msgs",       "span": "in_context"},
    {"what": "user prefers evening classes",    "kind": "fact",       "span": "long_term"},
    {"what": "last refund attempt failed - retry with an order id", "kind": "experience", "span": "long_term"},
    {"what": "always answer prices in INR",     "kind": "rule",       "span": "long_term"},
]
for it in items:
    kind, why = classify(it)
    print(f"  {it['what']:44s} -> {kind:11s} ({why})")

# Output:
#   the last 6 chat messages                     -> working     (the running turn, live in the context window)
#   user prefers evening classes                 -> semantic    (a durable fact or preference (long-term store))
#   last refund attempt failed - retry with an order id -> episodic    (a past event/outcome you can recall (long-term))
#   always answer prices in INR                  -> procedural  (a learned instruction the agent follows)

**Explanation**

A pure decision function, not a model - it is the taxonomy of the whole lesson expressed as branching rules. `classify` reads two tags on each item (its `kind` and its `span`) and returns which of the four memory types it is plus a one-line reason, showing that the type is decided by what the state IS, not by its content.

**How the code works - step by step**
- **`classify(item)`** - checks `span` first: anything `in_context` is **working** memory regardless of content; otherwise it branches on `kind`.
- **`kind == 'fact'`** -> **semantic** (a durable fact/preference), **`'experience'`** -> **episodic** (a past event/outcome), **`'rule'`** -> **procedural** (a learned instruction).
- **`items`** - four sample states: recent chat messages, a preference, a failed refund experience, and a standing pricing rule.
- The loop classifies each and prints the label plus the plain-English why.

**In one line:** in-context = working; fact = semantic; experience = episodic; rule = procedural.

**What you'll see:** Four aligned lines mapping each state item to its kind: the last 6 chat messages -> working, "prefers evening classes" -> semantic, the failed refund -> episodic, and "always answer prices in INR" -> procedural.

## 2 - Working memory: the buffer and its limit

The simplest memory is a buffer: keep the last N turns, drop the rest. It is fast and needs no store - it is exactly the in-loop history from 11.1 - but the context window is finite, so a fixed buffer *forgets* the moment a turn slides off the edge. This cell makes that failure concrete.

In [ ]:
# WORKING MEMORY = a sliding window over recent turns. Simple, but it FORGETS what falls out.
class BufferMemory:
    def __init__(self, window):
        self.window, self.turns = window, []
    def add(self, role, text):
        self.turns.append((role, text))
        self.turns = self.turns[-self.window:]     # keep only the last `window` turns
    def context(self):
        return " | ".join(f"{r}:{t}" for r, t in self.turns)

mem = BufferMemory(window=4)
script = [("user", "my name is Priya"), ("user", "I want GenAI"),
          ("user", "my budget is 15000"), ("user", "I know Python"),
          ("user", "what is my name?")]     # the intro has now slid out of a 4-turn window
for role, text in script:
    mem.add(role, text)

print("window =", mem.window, "| turns kept:", len(mem.turns))
print("context now:", mem.context())
answerable = any("name is" in t for _, t in mem.turns)
print("can it answer 'what is my name?' ->", answerable, "(the intro fell out of the window)")

# Output:
# window = 4 | turns kept: 4
# context now: user:I want GenAI | user:my budget is 15000 | user:I know Python | user:what is my name?
# can it answer 'what is my name?' -> False (the intro fell out of the window)

**Explanation**

A minimal sliding-window class that demonstrates forgetting by design. `BufferMemory` keeps only the last `window` turns; the demo feeds five turns into a window of four so the user's name (turn 1) has already fallen off by the time they ask for it, and the code proves the fact is simply not there to recall.

**How the code works - step by step**
- **`__init__`** - stores the window size and an empty list of turns.
- **`add`** - appends `(role, text)`, then slices to `self.turns[-window:]` so anything older than the last `window` turns is dropped.
- **`context`** - joins the kept turns into a single string for the prompt.
- The five-turn `script` overflows a 4-turn window; the `answerable` check scans the kept turns for "name is" and finds nothing.

**In one line:** keep the last N turns, and anything older is gone.

**What you'll see:** Prints `window = 4 | turns kept: 4`, a context string containing only the last four turns (the name is absent), and `can it answer 'what is my name?' -> False` - the intro fell out of the window.

## 3 - Summarization: compress, don't just drop

Dropping old turns loses everything in them; summarization keeps the gist. When the buffer overflows, you compress the oldest turns into a **running summary** and keep only the most recent turns verbatim - carrying names, numbers, and preferences forward at a fraction of the tokens. The key trick is carrying the *prior* summary forward so re-summarizing does not quietly drop facts.

In [ ]:
# SUMMARIZATION / COMPACTION: instead of dropping old turns, COMPRESS them into a running
# summary and keep only recent turns verbatim. Lossy, but it retains the key facts.
import re
def summarize(prior, turns):              # a scripted extractor (a real system uses an LLM)
    facts = [prior] if prior else []      # carry the running summary forward, then add new facts
    for _, t in turns:
        if "name is" in t:  facts.append("name=" + t.split("name is")[1].strip())
        if "budget" in t:   m = re.search(r"\d+", t); facts.append("budget=" + (m.group() if m else "?"))
        if "Python" in t:   facts.append("knows=Python")
    return "; ".join(dict.fromkeys(facts))   # dedupe, keep order

class SummaryMemory:
    def __init__(self, recent):
        self.recent, self.turns, self.summary = recent, [], ""
    def add(self, role, text):
        self.turns.append((role, text))
        if len(self.turns) > self.recent:                 # overflow -> compact the oldest
            old, self.turns = self.turns[:-self.recent], self.turns[-self.recent:]
            self.summary = summarize(self.summary, old)   # prior summary + facts from the dropped turns

mem = SummaryMemory(recent=2)
for role, text in [("user", "my name is Priya"), ("user", "my budget is 15000"),
                   ("user", "I know Python"), ("user", "recommend a course"),
                   ("user", "what is my name and budget?")]:
    mem.add(role, text)

print("running summary:", mem.summary)
print("recent verbatim:", " | ".join(t for _, t in mem.turns))
print("name+budget still known ->", "Priya" in mem.summary and "15000" in mem.summary)

# Output:
# running summary: name=Priya; budget=15000; knows=Python
# recent verbatim: recommend a course | what is my name and budget?
# name+budget still known -> True

**Explanation**

A buffer-plus-summary class that shows lossy compaction retaining the facts a raw buffer would have lost. On overflow it hands the dropped turns (plus the existing summary) to a scripted extractor and keeps only recent turns exact - the same tiered move production systems make, with an LLM standing in for the toy extractor.

**How the code works - step by step**
- **`summarize(prior, turns)`** - seeds the fact list with the prior summary (so nothing already compressed is lost), then pulls name/budget/Python facts from the dropped turns and dedupes while preserving order.
- **`SummaryMemory.__init__`** - holds `recent` (how many turns to keep verbatim), the turn list, and the running summary.
- **`add`** - appends the turn; when the list exceeds `recent`, it splits off the oldest turns and folds them into `self.summary`, keeping only the recent ones verbatim.
- The final check confirms name and budget survived in the summary.

**In one line:** compress the overflow into a running summary, keep recent turns exact.

**What you'll see:** Prints `running summary: name=Priya; budget=15000; knows=Python`, the two most recent turns verbatim, and `name+budget still known -> True` - compaction retained the key facts a raw buffer would have dropped.

## 4 - Semantic long-term memory: recall by meaning

Buffers and summaries still live inside one conversation. Semantic memory is the external store that survives *across sessions*: on write you extract a clean fact and embed it into a vector; on read you embed the query and return the closest facts by cosine similarity. This is the retrieval from Module 8 used as memory - and it is how assistants remember you days later, in a brand-new chat.

In [ ]:
# SEMANTIC LONG-TERM MEMORY: extract facts, EMBED them, recall by similarity - across sessions.
# The WRITE path also CONSOLIDATES: a near-duplicate UPDATES the old fact so memory does not go stale.
import numpy as np
VOCAB = ["python","developer","hyderabad","budget","15000","20000","rupees",
         "evening","classes","genai","name","priya"]
def embed(text):                          # TOY bag-of-words embedder (prod: a real embedding model)
    t = text.lower()
    v = np.array([1.0 if w in t else 0.0 for w in VOCAB])
    n = np.linalg.norm(v)
    return v / n if n else v

class VectorMemory:
    def __init__(self): self.facts = []
    def store(self, fact, thresh=0.7):
        fe = embed(fact)
        for i, (old, oe) in enumerate(self.facts):
            if float(np.dot(fe, oe)) >= thresh:   # same topic -> UPDATE (avoid stale duplicates)
                self.facts[i] = (fact, fe); return
        self.facts.append((fact, fe))             # new topic -> APPEND
    def recall(self, query, k=1):
        qe = embed(query)
        ranked = sorted(self.facts, key=lambda f: -float(np.dot(qe, f[1])))
        return [f[0] for f in ranked[:k]]

mem = VectorMemory()
for fact in ["Priya is a Python developer in Hyderabad",
             "Priya's budget is 15000 rupees",
             "Priya prefers evening classes"]:
    mem.store(fact)                       # written once; recallable days later, in a new session

for q in ["what is her budget in rupees?", "when does she like classes?", "is she a Python developer?"]:
    print(f"  '{q}' -> {mem.recall(q)}")

mem.store("Priya's budget is now 20000 rupees")   # a near-duplicate UPDATES the budget, not appends
print("after an update, facts stored:", len(mem.facts), "->", mem.recall("what is her budget?"))

# Output:
#   'what is her budget in rupees?' -> ["Priya's budget is 15000 rupees"]
#   'when does she like classes?' -> ['Priya prefers evening classes']
#   'is she a Python developer?' -> ['Priya is a Python developer in Hyderabad']
# after an update, facts stored: 3 -> ["Priya's budget is now 20000 rupees"]

**Explanation**

A tiny vector store that does both halves of semantic memory - similarity recall *and* consolidation. `store` embeds a fact and, crucially, updates a near-duplicate instead of appending a stale second copy; `recall` embeds the query and ranks by cosine similarity. The embedder is a toy bag-of-words so it runs with no key, but the mechanism is identical to production.

**How the code works - step by step**
- **`embed(text)`** - a toy bag-of-words embedder: a 0/1 vector over a fixed `VOCAB`, L2-normalized so dot product = cosine similarity.
- **`store(fact, thresh=0.7)`** - the write path: if a stored fact's similarity clears the threshold it **updates** that entry (consolidation, avoiding stale duplicates); otherwise it **appends** a new fact.
- **`recall(query, k=1)`** - the read path: embed the query, sort facts by descending cosine similarity, return the top `k`.
- The demo stores three facts, recalls each by *meaning*, then stores an updated budget - which overwrites the old one rather than adding a fourth fact.

**In one line:** extract a fact, embed it, recall by similarity - and update near-duplicates so memory stays fresh.

**What you'll see:** Each query retrieves the right fact by meaning (budget in rupees, evening classes, Python developer), then after the budget update it prints `facts stored: 3` and recall returns the current "...now 20000 rupees" - consolidation kept the count at 3.

## 5 - Episodic memory: learning from experience

Semantic memory stores *facts*; episodic memory stores *experiences* - what the agent tried, what happened, and a short reflection. On a later similar task the agent recalls those reflections and adapts, so it stops repeating the same failure. This is the Reflexion pattern from 11.2 as a memory system, and a recurring lesson can be promoted into a durable procedural rule.

In [ ]:
# EPISODIC MEMORY: store past ATTEMPTS (task, outcome, reflection) and recall them on a similar
# task, to learn from failure (Reflexion). A recurring lesson is then PROMOTED to a PROCEDURAL rule.
class EpisodicMemory:
    def __init__(self): self.episodes, self.rules = [], []
    def record(self, task, outcome, reflection):
        self.episodes.append({"task": task, "outcome": outcome, "reflection": reflection})
    def advice_for(self, task):
        key = task.split()[0].lower()      # crude task-type match (prod: embed the task)
        hits = [e for e in self.episodes if e["task"].split()[0].lower() == key]
        return [e["reflection"] for e in hits if e["outcome"] == "failed"]
    def promote(self, task):               # distil a recalled lesson into a durable procedural rule
        for r in self.advice_for(task):
            rule = "RULE: " + r
            if rule not in self.rules: self.rules.append(rule)

mem = EpisodicMemory()
mem.record("refund order #A1", "failed", "a refund needs the order id up front - ask for it first")
mem.record("refund order #B2", "success", "asked for the order id, then refunded")

new_task = "refund order #C3"
advice = mem.advice_for(new_task)          # EPISODIC recall
print("new task:", new_task)
print("recalled reflections:", advice)
print("acts differently this time ->", bool(advice))
mem.promote(new_task)                       # EPISODIC -> PROCEDURAL: the lesson becomes a standing rule
print("promoted to procedural rules:", mem.rules)

# Output:
# new task: refund order #C3
# recalled reflections: ['a refund needs the order id up front - ask for it first']
# acts differently this time -> True
# promoted to procedural rules: ['RULE: a refund needs the order id up front - ask for it first']

**Explanation**

An experience log that turns past failures into future behavior without any retraining. `record` stores attempts, `advice_for` recalls the reflections from *failed* attempts of the same task type, and `promote` distils a recalled lesson into a standing procedural rule - showing episodic experience hardening into procedural behavior.

**How the code works - step by step**
- **`__init__`** - two stores: `episodes` (the experience log) and `rules` (promoted procedural rules).
- **`record`** - appends an episode dict of task, outcome, and reflection.
- **`advice_for(task)`** - crude task-type match on the first word (prod: embed the task), returning reflections only from episodes whose outcome was `failed`.
- **`promote(task)`** - takes each recalled failure-lesson and adds it, deduped, to `rules` as a `RULE:` string - episodic becoming procedural.
- The demo records a failed and a successful refund, then a new refund task recalls the failure's lesson and promotes it.

**In one line:** log attempts + reflections, recall them on similar tasks, and promote recurring lessons to rules.

**What you'll see:** Prints the recalled reflection "a refund needs the order id up front - ask for it first", `acts differently this time -> True`, and a promoted procedural rule list containing that lesson prefixed with `RULE:`.

## 6 - Context engineering: what to actually load

You can store perfect memories and still fail, because the context window is finite *and* lossy. Models attend most to the start and end and under-attend the middle - the "lost in the middle" effect - so a bloated context can score worse than a focused one. Retrieval is only half the job; this cell models the other half: placing the fact that matters where the model will actually use it.

In [ ]:
# CONTEXT ENGINEERING: retrieval is only half. The context window is finite AND lossy -
# models attend to the START and END, not the MIDDLE ("lost in the middle"). Placement matters.
def recall_score(position, n):            # a MODEL of the U-shaped attention curve (illustrative)
    rel = position / (n - 1)              # 0.0 = first, 1.0 = last
    edge = min(rel, 1 - rel)              # distance from the nearer edge (0 at edges, 0.5 in middle)
    return round(1.0 - 1.4 * edge, 2)     # high at the edges, dips in the middle

n = 11
print("needle position -> recall (illustrative U-shape):")
for pos in (0, n // 2, n - 1):
    where = {0: "START", n // 2: "MIDDLE", n - 1: "END"}[pos]
    print(f"  {where:6s} (pos {pos:2d}/{n-1}) -> {recall_score(pos, n)}")

best = max(range(n), key=lambda p: recall_score(p, n))
print("fix: put the critical fact at the edges; a focused context beats a bloated one.")
print("best positions -> the edges (pos 0 and pos", n - 1, "), worst -> the middle")

# Output:
# needle position -> recall (illustrative U-shape):
#   START  (pos  0/10) -> 1.0
#   MIDDLE (pos  5/10) -> 0.3
#   END    (pos 10/10) -> 1.0
# fix: put the critical fact at the edges; a focused context beats a bloated one.
# best positions -> the edges (pos 0 and pos 10 ), worst -> the middle

**Explanation**

A measurement harness, not a model call - it computes an illustrative U-shaped attention curve to make placement concrete. `recall_score` maps a position in the context to a recall score that is high at the edges and dips in the middle, and the cell uses it to show that the same needle scores far better at the start or end than buried in the middle.

**How the code works - step by step**
- **`recall_score(position, n)`** - converts position to a 0-1 fraction, measures distance from the nearer edge, and returns `1.0 - 1.4 * edge` - high at edges, low in the middle (illustrative, not a fitted model).
- The loop scores three positions - START, MIDDLE, END - in an 11-slot context.
- **`best = max(...)`** - confirms the edges win, and the cell prints the fix: place critical facts at the edges and keep the context focused.

**In one line:** model the U-shaped curve, then place the needle at an edge, not the middle.

**What you'll see:** Prints the U-shape - START (1.0), MIDDLE (0.3), END (1.0) - then the fix line and "best positions -> the edges (pos 0 and pos 10), worst -> the middle."

Six mechanisms, one map: classify what a piece of state IS, and you know which memory to reach for. A buffer is fast working memory that forgets what slides out of the window; summarization compresses the overflow into a running summary; semantic vector recall gives durable, cross-session facts by meaning; and an episodic log lets the agent learn from its own past failures - which can then be promoted into a standing procedural rule. Context engineering is the read-side half everyone forgets: a finite, lossy window means you must select, compress, and place critical facts at the edges, not just retrieve them. Next, Lesson 11.5 needs *shared* memory across multiple agents, and Lesson 8.7 covers graph memory for multi-hop and temporal recall.